# Zadanie: klasyfikacja pozytywnego wyniku biopsji szyjki macicy

<div style="text-align: center;"><img src=".//Images//Zadanie.jpeg" alt="zadanie" width="400" height="120" style="margin: 10px; "/></div>

**Zbiór danych**: Cervical Cancer (Risk Factors)
[UCI ML Repository – Cervical Cancer](https://archive.ics.uci.edu/dataset/383/cervical+cancer+risk+factors)

- 858 przykładów
- 32 cechy (np. wiek, liczba partnerów seksualnych, palenie, itp.)
- 4 różne etykiety wyjściowe (binarnie oznaczone):
  - `Hinselmann`
  - `Schiller`
  - `Citology`
  - `Biopsy`

**Cel zadania:**  
Zbudować model klasyfikacyjny, który przewiduje wynik **`Biopsy`** (czy biopsja wykazała zmiany nowotworowe) i porównać skuteczność predykcji przy różnych metodach walidacji:

**Porównaj**:
- `KFold` vs. `StratifiedKFold`
- `ShuffleSplit` vs. `StratifiedShuffleSplit`

# Rozwiązanie

In [11]:
# Import bibliotek
import pandas as pd
import numpy as np

# Wczytanie danych
df = pd.read_csv("./Data/risk_factors_cervical_cancer.csv")

In [33]:
%run tools.ipynb

In [23]:
df.head(10)

,Age,Number of sexual partners,First sexual intercourse,Num of pregnancies,Smokes,Smokes (years),Smokes (packs/year),Hormonal Contraceptives,Hormonal Contraceptives (years),IUD,...,STDs: Time since first diagnosis,STDs: Time since last diagnosis,Dx:Cancer,Dx:CIN,Dx:HPV,Dx,Hinselmann,Schiller,Citology,Biopsy
0,18,4.0,15.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,?,?,0,0,0,0,0,0,0,0
1,15,1.0,14.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,?,?,0,0,0,0,0,0,0,0
2,34,1.0,?,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,?,?,0,0,0,0,0,0,0,0
3,52,5.0,16.0,4.0,1.0,37.0,37.0,1.0,3.0,0.0,...,?,?,1,0,1,0,0,0,0,0
4,46,3.0,21.0,4.0,0.0,0.0,0.0,1.0,15.0,0.0,...,?,?,0,0,0,0,0,0,0,0
5,42,3.0,23.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,?,?,0,0,0,0,0,0,0,0
6,51,3.0,17.0,6.0,1.0,34.0,3.4,0.0,0.0,1.0,...,?,?,0,0,0,0,1,1,0,1
7,26,1.0,26.0,3.0,0.0,0.0,0.0,1.0,2.0,1.0,...,?,?,0,0,0,0,0,0,0,0
8,45,1.0,20.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,...,?,?,1,0,1,1,0,0,0,0
9,44,3.0,15.0,?,1.0,1.266972909,2.8,0.0,0.0,?,...,?,?,0,0,0,0,0,0,0,0


In [ ]:
df.replace('?', np.nan, inplace=True)
df = df.astype(float)

df.fillna(df.median(), inplace=True)

df['Biopsy'].value_counts()


Biopsy
0.0    803
1.0     55
Name: count, dtype: int64

In [40]:
X = df.drop(columns=['Hinselmann', 'Schiller', 'Citology', 'Biopsy']).values
y = df['Biopsy'].values.astype(int)

print("X:", X.shape)
print("y - rozkład klas:", np.bincount(y))


X: (858, 32)
y - rozkład klas: [803  55]


In [38]:
# KFold vs StratifiedKFold
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)

kf  = KFold(n_splits=5, shuffle=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores_kf  = cross_val_score(model, X, y, cv=kf,  scoring='f1')
scores_skf = cross_val_score(model, X, y, cv=skf, scoring='f1')

print("KFold            F1: {:.4f} (+/- {:.4f})".format(scores_kf.mean(),  scores_kf.std()))
print("StratifiedKFold  F1: {:.4f} (+/- {:.4f})".format(scores_skf.mean(), scores_skf.std()))


KFold            F1: 0.1397 (+/- 0.0746)
StratifiedKFold  F1: 0.0756 (+/- 0.0971)


In [39]:
# ShuffleSplit vs StratifiedShuffleSplit
from sklearn.model_selection import ShuffleSplit, StratifiedShuffleSplit

ss  = ShuffleSplit(n_splits=10, test_size=0.2, random_state=42)
sss = StratifiedShuffleSplit(n_splits=10, test_size=0.2, random_state=42)

scores_ss  = cross_val_score(model, X, y, cv=ss,  scoring='f1')
scores_sss = cross_val_score(model, X, y, cv=sss, scoring='f1')

print("ShuffleSplit            F1: {:.4f} (+/- {:.4f})".format(scores_ss.mean(),  scores_ss.std()))
print("StratifiedShuffleSplit  F1: {:.4f} (+/- {:.4f})".format(scores_sss.mean(), scores_sss.std()))


ShuffleSplit            F1: 0.0452 (+/- 0.0694)
StratifiedShuffleSplit  F1: 0.0487 (+/- 0.0745)
